# Практическое занятие № 16 — кластеризация заказов методом K-means
**ТюмГУ · «Технологии искусственного интеллекта — 1» · 90 минут**

**Цель:** выделить группы похожих заказов по признакам, доступным **в момент создания заказа**, сравнить несколько K, интерпретировать центроиды и проверить устойчивость.

`actual_minutes` и `late` НЕ используются для обучения K-means, выбора K и масштабирования. K-means не предсказывает «верные классы»: номера кластеров — условные. `demo_orders_13_14.csv` нужен только для проверки работоспособности.

## 1. Подготовка данных и постановка задачи (0–20 мин)
Используем три признака, доступных до доставки: `distance_km`, `queue_size`, `promised_minutes`. Стандартизация необходима из-за разных масштабов. Сохраняем хронологический holdout 20%: подбор K только на первых 80% заказов.

In [ ]:
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score,calinski_harabasz_score,davies_bouldin_score,adjusted_rand_score
from sklearn.base import clone

DATA_PATH=Path('orders_dirty.csv')
if not DATA_PATH.exists(): DATA_PATH=Path('demo_orders_13_14.csv')
if not DATA_PATH.exists(): raise FileNotFoundError('Загрузите orders_dirty.csv или демонстрационный demo_orders_13_14.csv')
print('ИСТОЧНИК:', DATA_PATH.name, '(демо — НЕ основание для выводов об исходных заказах)' if DATA_PATH.name.startswith('demo') else '')
raw=pd.read_csv(DATA_PATH,encoding='utf-8-sig')
FEATURES=['distance_km','queue_size','promised_minutes']
assert {'order_id','created_at',*FEATURES}.issubset(raw.columns)
source_size=len(raw); duplicates=int(raw.duplicated().sum())
df=raw.drop_duplicates().copy()
df['created_at']=pd.to_datetime(df['created_at'],errors='coerce')
df=df.dropna(subset=['order_id','created_at'])
if not df['order_id'].is_unique:
    raise ValueError('Повторные order_id с различными строками: решите конфликт ключей перед обучением!')
for c in FEATURES:
    df[c]=pd.to_numeric(df[c],errors='coerce')
    df.loc[df[c]<0,c]=np.nan
# promised_minutes — положительная длительность; ноль физически недопустим.
df.loc[df['promised_minutes']<=0,'promised_minutes']=np.nan
df=df.sort_values(['created_at','order_id'],kind='stable').reset_index(drop=True)
assert len(df)>=30,'Для этой практики нужно не менее 30 корректно датированных заказов.'
n_dev=int(len(df)*.8)
dev=df.iloc[:n_dev].copy(); holdout=df.iloc[n_dev:].copy()
assert set(dev.order_id).isdisjoint(set(holdout.order_id))
assert dev.created_at.max()<=holdout.created_at.min()
print('Строк исходно:',source_size,'полных повторов:',duplicates,'в работе:',len(df))
print('Dev / Holdout:',len(dev),len(holdout))
print('Входы:',FEATURES,'; исходные late и actual_minutes в них отсутствуют')
print('Пропуски в dev:\n',dev[FEATURES].isna().sum())
assert {'late','actual_minutes','order_id'}.isdisjoint(FEATURES)

prep=Pipeline([('impute',SimpleImputer(strategy='median',keep_empty_features=True)),
               ('scale',StandardScaler())])
Z=prep.fit_transform(dev[FEATURES])  # только dev! Не holdout.
assert Z.shape==(len(dev),len(FEATURES)) and np.isfinite(Z).all()
print('Средние нормализованных признаков:',np.mean(Z,axis=0).round(3))

## 2. K-means: алгоритм и зависимость от K (20–45 мин)
Минимизируем сумму квадратов расстояний до центроидов (inertia):  \(J=\sum_{i=1}^n\|z_i-\mu_{c_i}\|^2\). 
Каждый кандидат использует одинаковую матрицу `Z` и `random_state=42`. Silhouette максимизируем, DB минимизируем; никакой показатель не доказывает существование «естественных» групп.

In [ ]:
ks=list(range(2,min(7,len(dev)-1)))
assert len(ks)>=2
rows=[]; fitted={}
for k in ks:
    km=KMeans(n_clusters=k,init='k-means++',n_init=10,random_state=42)
    lab=km.fit_predict(Z)
    rows.append({'k':k,'inertia':float(km.inertia_),
                 'silhouette':float(silhouette_score(Z,lab)),
                 'calinski_harabasz':float(calinski_harabasz_score(Z,lab)),
                 'davies_bouldin':float(davies_bouldin_score(Z,lab))})
    fitted[k]=km
scores=pd.DataFrame(rows)
print(scores.round(3).to_string(index=False))
scores.to_csv('16_k_metrics.csv',index=False,encoding='utf-8-sig')
fig,axs=plt.subplots(1,2,figsize=(10.5,3.9))
axs[0].plot(scores.k,scores.inertia,'o-');axs[0].set(xlabel='K',ylabel='Inertia (SSE)',title='Elbow: сумма квадратов расстояний')
axs[1].plot(scores.k,scores.silhouette,'o-');axs[1].set(xlabel='K',ylabel='Silhouette',title='Средняя ширина силуэта')
fig.tight_layout();fig.savefig('16_elbow_silhouette.png',dpi=160);plt.show()
# Явное учебное правило выбора, без late / actual_minutes.
chosen_k=int(scores.loc[scores.silhouette.idxmax(),'k'])
print('K по максимуму Silhouette:',chosen_k,'(проинтерпретируйте также Elbow и бизнес-смысл)')

## 3. Устойчивость к инициализации и масштабам (45–65 мин)
Повторите обучение при разных seed и сравните разметки через Adjusted Rand Index (ARI): он устойчив к переименованию кластеров. Сравните K-means на числах без стандартизации **только как контрольный эксперимент**; SSE разных шкал нельзя сравнивать напрямую.

In [ ]:
from itertools import combinations
labels_by_seed=[]
for seed in [11,21,31,41,51]:
    labels_by_seed.append(KMeans(n_clusters=chosen_k,init='k-means++',n_init=10,
                                 random_state=seed).fit_predict(Z))
ari=[adjusted_rand_score(a,b) for a,b in combinations(labels_by_seed,2)]
print('ARI при разных seed: среднее',round(float(np.mean(ari)),3),'минимум',round(float(np.min(ari)),3))
# В исходной шкале километры, минуты и очередь имеют разный вес в евклидовой метрике.
raw_imputed=SimpleImputer(strategy='median').fit_transform(dev[FEATURES])
raw_labels=KMeans(n_clusters=chosen_k,n_init=10,random_state=42).fit_predict(raw_imputed)
scaled_labels=fitted[chosen_k].labels_
print('ARI масштабировано vs без масштаба:',round(float(adjusted_rand_score(raw_labels,scaled_labels)),3))
stab=pd.DataFrame({'metric':['mean_pairwise_ARI','min_pairwise_ARI','scaled_vs_raw_ARI'],
                   'value':[float(np.mean(ari)),float(np.min(ari)),float(adjusted_rand_score(raw_labels,scaled_labels))]})
stab.to_csv('16_stability.csv',index=False,encoding='utf-8-sig')

## 4. Профили кластеров и визуализация (65–82 мин)
Центроиды переводятся обратно в исходные единицы через `inverse_transform`. График показывает только две координаты, хотя модель обучалась на трёх; точки на проекции могут перекрываться без совпадения в полном пространстве признаков. Отдельно укажите количество заказов в каждом кластере.

In [ ]:
km=fitted[chosen_k]
profiles=dev[FEATURES].copy()
profiles['cluster']=km.labels_
profiles['order_id']=dev.order_id.values
counts=profiles.groupby('cluster').size().rename('n')
means=profiles.groupby('cluster')[FEATURES].mean()
centers=pd.DataFrame(prep.named_steps['scale'].inverse_transform(km.cluster_centers_),columns=FEATURES)
centers.insert(0,'cluster',np.arange(chosen_k))
print('Профили групп:');print(pd.concat([counts,means],axis=1).round(2).to_string())
print('\nЦентроиды в исходных единицах:\n',centers.round(2).to_string(index=False))
centers.to_csv('16_centroids.csv',index=False,encoding='utf-8-sig')
profiles.to_csv('16_cluster_assignments_dev.csv',index=False,encoding='utf-8-sig')
fig,ax=plt.subplots(figsize=(8,5))
scatter=ax.scatter(dev['distance_km'],dev['queue_size'],c=km.labels_,cmap='tab10',s=30,alpha=.76)
ax.set(xlabel='Расстояние, км',ylabel='Очередь, заказов',title='Кластеры по 3 признакам (показана проекция 2D)')
ax.legend(*scatter.legend_elements(),title='Номер группы',loc='best')
fig.tight_layout();fig.savefig('16_clusters_projection.png',dpi=160);plt.show()
# Присвоение меток новых заказов без подбора числа кластеров по holdout.
Z_holdout=prep.transform(holdout[FEATURES]);held_labels=km.predict(Z_holdout)
pd.DataFrame({'order_id':holdout.order_id.values,'cluster':held_labels}).to_csv('16_cluster_assignments_holdout.csv',index=False,encoding='utf-8-sig')
print('Holdout использован только для присвоения меток, не для подбора K.')

## 5. Отчёт и самостоятельная работа (82–90 мин)
1. Дайте смысловое название каждому кластеру по **трём признакам** и его размеру. Не называйте группы «плохими» или «хорошими» без внешней проверки.
2. Покажите Elbow, Silhouette, CH, DB и обоснуйте свой K. Перечислите по одной ситуации, когда каждый индекс вводит в заблуждение.
3. Объясните, почему стандартизация влияет на расстояния и распределение кластеров; приведите ARI.
4. Объясните, почему результаты K-means не являются доказательством причин задержек или готовой сегментацией клиентов.
5. Дополнительная часть: добавьте категориальный `weather` через OneHotEncoder, сопоставьте устойчивость; обязательно обсудите изменение весов блоков признаков.

**Сдать:** `Группа_Фамилия_З16_Отчет.pdf` (или DOCX) + `16_student.ipynb` + таблицы `16_k_metrics.csv`, `16_centroids.csv`, `16_stability.csv`. Блокнот должен работать после Restart & Run All.